In [ ]:
%pip install comet_ml > /dev/null 2>&1
%pip install mido
%pip install torch
%pip install tqdm
%pip install IPython
%pip install pyyaml

from src.dataset_processing import dataset_processing
from src.model import make_transformer_vae, TransformerVAE
from src.generation import continue_text, continue_text_with_interpolation
from src.tokens2midi import token2midi

from .load_checkpoint import load_checkpoint

import yaml
import torch

### Processing the dataset ###
training_set_path = "PUT YOUR TOKENIZED TRAINING SET PATH HERE" # e.g., "datasets/CPRemi/LateRomantic/Training-Set"
validation_set_path = "PUT YOUR TOKENIZED VALIDATION SET PATH HERE" # e.g., "datasets/CPRemi/LateRomantic/Validation-Set"

vectorized_songs, validation_set, field2idx, idx2field, vocab_sizes = dataset_processing(training_set_path, validation_set_path)

### Creating the model and loading checkpoint ###
config_path = "PUT YOUR CONFIG FILE PATH HERE" # e.g., "configs/config1.yaml"

with open(config_path, "r") as f:
    params = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model = make_transformer_vae(params, vocab_sizes)
my_model.to(device)

checkpoint_dir = "PUT YOUR CHECKPOINT DIRECTORY PATH HERE" # e.g., "ckpts/100k"

# You may have no saved .pt files in your ckpts directory yet, in which case the model uses randomly initialized weights.

load_checkpoint(my_model, checkpoint_dir, use_best_ckpt=True, device=device) # Set use_best_ckpt to True to load the best checkpoint, or False to load the latest checkpoint

# Alternatively, you can make a model without creating your own config file by specifying the parameters directly in a dictionary. For example:

#params = dict(
#    num_training_iterations=10000,
#    epochs=10,
#    batch_size=8,
#    seq_length=1024,
#    learning_rate=2e-4,
#    d_model=256,
#    num_heads=8,
#    dropout=0.1,
#    num_layers=4,
#   latent_dim=64,
#    attribute_embedding_dim=32,
#    max_beta=0.1,
#    ratio_zero=0.1,
#    ratio_increase=0.1
#)

#model = TransformerVAE(
#    vocab_sizes=vocab_sizes,
#    latent_dim=params["latent_dim"],
#    attribute_embedding_dim=params["attribute_embedding_dim"],
#    block_size=params["seq_length"],
#    d_model=params["d_model"],
#    num_heads=params["num_heads"],
#    dropout=params["dropout"],
#    num_layers=params["num_layers"]
#)

### Generating music with the model ###

out_directory = "PUT YOUR OUTPUT DIRECTORY PATH HERE" # e.g., "generated/generated_tokens/generated_text" <- note that you're naming the .txt file here
prompt_path = "PUT YOUR TOKENIZED PROMPT FILE PATH HERE" # e.g., "datasets/CPRemi/LateRomantic/Validation-Set/Beethoven_Symphony_9_1_1.txt"

bar_start = 9 # Specify the starting bar for generation
generation_length = 1024 # Specify the desired length of the generated sequence
batch_size = 3 # Specify the batch size for generation
sample_latent = True # Specify whether to sample from the latent space or use the mean

# Continuation without interpolation

continue_text(
    model=my_model,
    out_path=out_directory,
    prompt_path=prompt_path,
    bar_start=bar_start,
    generation_length=generation_length,
    batch_size=batch_size,
    params=params,
    device="cuda" if torch.cuda.is_available() else "cpu",
    field2idx=field2idx,
    idx2field=idx2field,
    sample_latent=sample_latent
)

# You need two prompts for interpolation: One for the start and one for the end of the interpolation. The prompts should be tokenized text files.
encoded_start_prompt_path = "PUT YOUR TOKENIZED START PROMPT FILE PATH HERE" # e.g., sampled_bars/CPRemi/mahlers-8th-symphony-finalef29t44.txt
encoded_end_prompt_path = "PUT YOUR TOKENIZED END PROMPT FILE PATH HERE" # e.g., sampled_bars/CPRemi/mahlers-8th-symphony-finalef125t152.txt

interpolation_start_bar = 9 # Specify the starting bar for the interpolation
interpolation_length_bars = 24 # Specify the desired length of the interpolation in bars

# Continuation with interpolation

continue_text_with_interpolation(
    model=my_model,
    out_path=out_directory,
    encoded_prompt1=encoded_start_prompt_path,
    encoded_prompt2=encoded_end_prompt_path,
    generation_length=generation_length,
    batch_size=batch_size,
    params=params,
    device=device,
    field2idx=field2idx,
    idx2field=idx2field,
    interpolation_start_bar=interpolation_start_bar,
    interpolation_length_bars=interpolation_length_bars,
    sample_latent=sample_latent
)

generated_path = "YOUR GENERATED MIDI OUTPUT DIRECTORY PATH HERE" # e.g., "generated/generated_midis/generated_mid" <- note that you're naming the .mid file here
for i in range(batch_size):
  token2midi(out_directory + f"{i}.txt", generated_path + f"{i}.mid")
